In [1]:
import tensorflow as tf 
import re
import numpy as np 
from sklearn.model_selection import train_test_split

In [5]:
def preprocess_sentence(w):
    w = w.lower() # lowercase sentence
    w = re.sub(r"([?.!,])", r"\1 ", w) # tambah space setelah tanda baca
    w = re.sub(r'[" "]+', " ", w) # apus spasi yang berlebihan
    w = re.sub(r'[^a-zA-Z?.!,]+', " ", w) # apus karakter selain alphanumeric
    w = w.strip() # apus spasi di awal + di akhir
    w = '<start> ' + w + ' <end>'
    return w 

def load_dataset(path, num_examples=None):
    with open(path, "r", encoding="utf-8") as file:
        lines = file.read().strip().split("\n") # read, buang spasi, split per new line
    
    parts = [line.split("\t") for line in lines [:num_examples]]
    input_data = [preprocess_sentence(part[0]) for part in parts]
    target_data = [preprocess_sentence(part[1]) for part in parts]
    
    return input_data, target_data

path_to_file = "./ind.txt"
input_data, target_data = load_dataset(path_to_file, num_examples=100000)
print(f"Input Data: {input_data}")
print(f"Target Data: {target_data}")

Input Data: ['<start> hi. <end>', '<start> run! <end>', '<start> run. <end>', '<start> who? <end>', '<start> wow! <end>', '<start> wow! <end>', '<start> fire! <end>', '<start> fire! <end>', '<start> fire! <end>', '<start> help! <end>', '<start> jump! <end>', '<start> jump. <end>', '<start> stop! <end>', '<start> stop! <end>', '<start> wait! <end>', '<start> wait. <end>', '<start> hello. <end>', '<start> hurry! <end>', '<start> hurry! <end>', '<start> hurry! <end>', '<start> i see. <end>', '<start> i see. <end>', '<start> i see. <end>', '<start> i see. <end>', '<start> i see. <end>', '<start> i see. <end>', '<start> i won! <end>', '<start> i won. <end>', '<start> oh no! <end>', '<start> relax. <end>', '<start> relax. <end>', '<start> shoot! <end>', '<start> smile. <end>', '<start> attack! <end>', '<start> cheers! <end>', '<start> freeze! <end>', '<start> get up. <end>', '<start> got it! <end>', '<start> got it? <end>', '<start> he ran. <end>', '<start> i fell. <end>', '<start> i know. <

In [14]:
# Tokenize input data dan target data jadi angka / vektor
tokenizer = tf.keras.preprocessing.text.Tokenizer(filters='')  
tokenizer.fit_on_texts(input_data + target_data)

# Convert kalimat ke integer sequences
input_tensor = tokenizer.texts_to_sequences(input_data)
target_tensor = tokenizer.texts_to_sequences(target_data)

# Calculate max sequence length
max_length_target, max_length_input = max(len(seq) for seq in target_tensor), max(len(seq) for seq in input_tensor)

# Pad sequences
input_tensor = tf.keras.preprocessing.sequence.pad_sequences(input_tensor, maxlen=max_length_input, padding="post")
target_tensor = tf.keras.preprocessing.sequence.pad_sequences(target_tensor, maxlen=max_length_target, padding="post") # Padding samain vektor yang lebih pendek, tambahin 0

vocab_size = len(tokenizer.word_index) + 1
print(f"Input Tensor: {input_tensor}")
print(f"Target Tensor: {target_tensor}")
print(tokenizer.word_index)

Input Tensor: [[    1  4450     2 ...     0     0     0]
 [    1  8571     2 ...     0     0     0]
 [    1  5763     2 ...     0     0     0]
 ...
 [    1 12056 12057 ...     0     0     0]
 [    1  7194  7195 ...     0     0     0]
 [    1   152    11 ...    27  2522     2]]
Target Tensor: [[    1  5085     2 ...     0     0     0]
 [    1  7197     2 ...     0     0     0]
 [    1  7197     2 ...     0     0     0]
 ...
 [    1 16335 16336 ...     0     0     0]
 [    1  7194  7195 ...     0     0     0]
 [    1   277   614 ...     0     0     0]]
{'<start>': 1, '<end>': 2, 'tom': 3, 'i': 4, 'the': 5, 'aku': 6, 'you': 7, 'to': 8, 'is': 9, 'tidak': 10, 'a': 11, 't': 12, 'yang': 13, 'di': 14, 'kamu': 15, 's': 16, 'saya': 17, 'dia': 18, 'he': 19, 'itu': 20, 'in': 21, 'apa': 22, 'this': 23, 'ini': 24, 'my': 25, 'akan': 26, 'that': 27, 'of': 28, 'do': 29, 'have': 30, 'it': 31, 'don': 32, 'what': 33, 'was': 34, 'we': 35, 'are': 36, 'your': 37, 'm': 38, 'ada': 39, 'bisa': 40, 'can': 41, 'm

In [ ]:
# Split data
input_tensor_train, input_tensor_val, target_tensor_train, target_tensor_val = train_test_split(
    input_tensor, target_tensor, test_size=0.2, random_state=69
)

BUFFER_SIZE = len(input_tensor_train)
BATCH_SIZE = 64

dataset = tf.data.Dataset.from_tensor_slices((input_tensor_train, target_tensor_train)).shuffle(BUFFER_SIZE)
dataset = dataset.batch(BATCH_SIZE, drop_remainder = True)
print(f"Dataset: {dataset}")
for sample in dataset.take(1):
    input_sample, target_sample = sample
    print(f"Input Example: {input_sample}")
    print(f"Target Example: {target_sample}")

Dataset: <BatchDataset element_spec=(TensorSpec(shape=(64, 36), dtype=tf.int32, name=None), TensorSpec(shape=(64, 30), dtype=tf.int32, name=None))>
Input Example: [[   1    3  559 ...    0    0    0]
 [   1   82  221 ...    0    0    0]
 [   1    4  619 ...    0    0    0]
 ...
 [   1    3 1461 ...    0    0    0]
 [   1   19  120 ...    0    0    0]
 [   1    4  239 ...    0    0    0]]
Target Example: [[   1    3 2655 ...    0    0    0]
 [   1  150 2124 ...    0    0    0]
 [   1    6  627 ...    0    0    0]
 ...
 [   1    3  644 ...    0    0    0]
 [   1   18   10 ...    0    0    0]
 [   1    6   10 ...    0    0    0]]


In [42]:
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, enc_units, batch_size):
        super(Encoder, self).__init__()
        self.batch_size = batch_size
        self.enc_units = enc_units
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.gru = tf.keras.layers.GRU(
            self.enc_units,
            return_sequences=True,
            return_state=True,
            recurrent_initializer="glorot_uniform"
        )
    
    def call(self, x, hidden):
        x = self.embedding(x)
        output, state = self.gru(x, initial_state=hidden)
        return output, state
    
    def initialize_hidden_state(self):
        return tf.zeros((self.batch_size, self.enc_units))

embedding_dim = 256
units = 1024
BATCH_SIZE = 64
example_input_batch = input_tensor_train[:BATCH_SIZE]
encoder = Encoder(vocab_size, embedding_dim, units, BATCH_SIZE)

sample_hidden = encoder.initialize_hidden_state()
sample_output, sample_hidden = encoder(example_input_batch, sample_hidden)

print(f"Encoder output shape (batch size, seq length, units): {format(sample_output.shape)}")
print(f"Encoder hidden shape (batch size, units): {format(sample_hidden.shape)}")

Encoder output shape (batch size, seq length, units): (64, 36, 1024)
Encoder hidden shape (batch size, units): (64, 1024)
